# 03 — Run Simulation
**Wildfire Risk Modeling Exercise | Cell2Fire W — Scott & Burgan**

Executes Cell2Fire W using the instance folder produced by notebook 02.

- `nsims=1` → deterministic single run
- Increase `nsims` in `config/<town>.yaml` for burn probability
- All parameters read from config — change `TOWN` to run Prairie

In [ ]:
TOWN = "forest"   # "forest" or "prairie"

In [ ]:
# ============================================================
# CELL 1 — Imports, config, paths
# ============================================================
import sys, pathlib, subprocess, time, shutil
import yaml

REPO_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils import load_config

cfg      = load_config(TOWN, REPO_ROOT)
INSTANCE = REPO_ROOT / cfg["instance_dir"]
RESULTS  = REPO_ROOT / cfg["results_dir"]
RESULTS.mkdir(parents=True, exist_ok=True)
sim_cfg  = cfg["simulation"]

BINARY   = REPO_ROOT / "C2F-W" / "Cell2Fire" / "Cell2Fire"

# Verify instance is ready
required = ["fuels.asc", "elevation.asc", "Weather.csv",
            "Ignitions.csv", "spain_lookup_table.csv", "Data.csv"]
all_ok = True
print(f"Town:     {cfg['display_name']}")
print(f"Binary:   {'✓' if BINARY.exists() else '✗ MISSING'}")
print(f"nsims:    {sim_cfg['nsims']}")
print()
for fname in required:
    ok = (INSTANCE / fname).exists()
    print(f"  {'✓' if ok else '✗'}  {fname}")
    if not ok: all_ok = False
print()
print("✓ Ready to simulate" if all_ok else "✗ Run notebook 02 first")

In [ ]:
# ============================================================
# CELL 2 — Run Cell2Fire W
# ============================================================
shutil.rmtree(RESULTS, ignore_errors=True)
RESULTS.mkdir()

cmd = [
    str(BINARY),
    "--input-instance-folder", str(INSTANCE),
    "--output-folder",         str(RESULTS),
    "--sim",             sim_cfg["sim_model"],
    "--nsims",           str(sim_cfg["nsims"]),
    "--nthreads",        str(sim_cfg["nthreads"]),
    "--weather",         sim_cfg["weather_mode"],
    "--ignitions",
    "--grids",
    "--final-grid",
    "--output-messages",
    "--Fire-Period-Length", "1.0",
    "--ROS-CV",          "0.0",
    "--seed",            str(sim_cfg["seed"]),
]

print("── Simulation parameters ────────────────────────────")
print(f"  Model   : Scott & Burgan (--sim {sim_cfg['sim_model']})")
print(f"  nsims   : {sim_cfg['nsims']}")
print(f"  Threads : {sim_cfg['nthreads']}  |  Seed: {sim_cfg['seed']}")
print(f"  Duration: {sim_cfg['duration_hours']} hours")
print(f"\nRunning...")

t0      = time.time()
result  = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0

# Save log
log_path = RESULTS / "run.log"
log_path.write_text(result.stdout + "\n" + result.stderr)

print(f"  Exit code : {result.returncode}  ({elapsed:.1f}s)")
print(f"  Log saved : {log_path}")

if result.returncode != 0:
    print("\nERROR:")
    print(result.stderr[-1000:])
    raise RuntimeError("Simulation failed — check log above")

# Print summary lines from stdout
print(f"\n── Simulation output ────────────────────────────────")
keywords = ["Simulation", "Cell Status", "---", "Available",
            "Burnt", "Non-Burnable", "Total", "ignition",
            "max time", "weather"]
for line in result.stdout.splitlines():
    if any(k in line for k in keywords):
        print(f"  {line.strip()}")

# Count output files
grid_dir = RESULTS / "Grids" / "Grids1"
n_grids  = len(list(grid_dir.glob("*.csv"))) if grid_dir.exists() else 0
msg_ok   = (RESULTS / "Messages" / "MessagesFile1.csv").exists()

print(f"\n── Output files ─────────────────────────────────────")
print(f"  Grid snapshots : {n_grids} timestep files")
print(f"  Messages file  : {'✓' if msg_ok else '✗'}")
print(f"  Runtime        : {elapsed:.1f}s")

print(f"\n✓ Cell 2 ready — simulation complete, open notebook 04 for results")